# GPU quickstart

A 60-second check that the GPU is real and usable, then a small benchmark.

Your slice has 18 GB of VRAM. Sizes below stay well inside that.


In [ ]:
import torch

assert torch.cuda.is_available(), 'No GPU on this profile — restart with a GPU profile.'
dev = torch.device('cuda')
print(torch.cuda.get_device_name(0))


## A real matmul, timed

`torch.cuda.synchronize()` matters here: CUDA calls are asynchronous, so timing
without it measures how fast Python queues work, not how fast the GPU does it.


In [ ]:
import time

a = torch.randn(8192, 8192, device=dev)
b = torch.randn(8192, 8192, device=dev)

torch.cuda.synchronize()
t0 = time.perf_counter()
for _ in range(10):
    c = a @ b
torch.cuda.synchronize()
dt = (time.perf_counter() - t0) / 10

flops = 2 * 8192**3
print(f'{dt*1000:.1f} ms per matmul  ->  {flops/dt/1e12:.1f} TFLOP/s')


## Watch your VRAM

The single most common failure here is an out-of-memory error partway through
training. Check headroom before scaling a job up.


In [ ]:
free, total = torch.cuda.mem_get_info()
print(f'VRAM free: {free/2**30:.1f} GB of {total/2**30:.1f} GB')

del a, b, c
torch.cuda.empty_cache()
free, _ = torch.cuda.mem_get_info()
print(f'after cleanup: {free/2**30:.1f} GB free')
